# Interview Deception Miner Summary

Loads `deception_samples.jsonl` files from the interview miner results and gives an interactive browser for deceptive examples.


In [ ]:
from pathlib import Path
import html
import json

import ipywidgets as widgets
import pandas as pd
from IPython.display import HTML, JSON, Markdown, clear_output, display

pd.set_option('display.max_colwidth', 200)

repo_root = Path('/playpen-ssd/smerrill/deception2')
data_root = repo_root / 'Interview' / 'Results' / 'deception_miner' / 'deepseek-ai_DeepSeek-R1-Distill-Qwen-7B'
data_root


In [ ]:
jsonl_files = sorted(data_root.rglob('deception_samples.jsonl'))
print(f'Found {len(jsonl_files)} jsonl files under {data_root}')

def exact_action_type(value):
    if value is None:
        return None
    text = str(value).strip()
    return text if text else None


VALID_ACTION_TYPES = {'ACCEPT', 'REJECT', 'REJECT_CLAIM_HIGHER_OFFER'}

records = []
for path in jsonl_files:
    run_tag = path.parent.parent.name
    shard = path.parent.name
    meta_path = path.parent / 'meta.json'
    meta = {}
    if meta_path.exists():
        try:
            meta = json.loads(meta_path.read_text(encoding='utf-8'))
        except Exception:
            meta = {}

    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
            except Exception:
                continue

            action = rec.get('action', {}) if isinstance(rec.get('action'), dict) else {}
            truth_context = rec.get('truth_context', {}) if isinstance(rec.get('truth_context'), dict) else {}
            label_preview = truth_context.get('label_preview', {}) if isinstance(truth_context.get('label_preview'), dict) else {}
            reasoning = action.get('reasoning') or action.get('_raw_text')
            message = action.get('Message')
            action_type = exact_action_type(action.get('Action'))
            recorded_deceptive = rec.get('deceptive')
            label_preview_deceptive = label_preview.get('deceptive')
            if action_type in VALID_ACTION_TYPES:
                deceptive = recorded_deceptive if recorded_deceptive in (True, False) else label_preview_deceptive
            else:
                deceptive = None
            deceptive_label = 'DECEPTIVE' if deceptive is True else ('TRUTHFUL' if deceptive is False else 'UNKNOWN')

            records.append({
                'source_path': str(path),
                'run_tag': run_tag,
                'shard': shard,
                'model_name': meta.get('model_name'),
                'conversation_id': rec.get('conversation_id'),
                'scenario_name': rec.get('scenario_name'),
                'base_scenario_name': rec.get('base_scenario_name'),
                'state_id': rec.get('state_id'),
                'game_id': rec.get('game_id'),
                'turn_idx': rec.get('turn_idx'),
                'sample_idx': rec.get('sample_idx'),
                'seed': rec.get('seed'),
                'deceptive': deceptive,
                'recorded_deceptive': recorded_deceptive,
                'deceptive_label': deceptive_label,
                'used_for_transition': rec.get('used_for_transition'),
                'action_type': action_type,
                'claimed_other_offer_salary': action.get('Claimed_other_offer_salary'),
                'message': message,
                'reasoning': reasoning,
                'decision_message_consistent': label_preview.get('decision_message_consistent'),
                'message_mentions_other_offer_claim': label_preview.get('message_mentions_other_offer_claim'),
                'message_mentions_claimed_other_offer_salary': label_preview.get('message_mentions_claimed_other_offer_salary'),
                'deception_reason': label_preview.get('deception_reason'),
                'full_action': action,
                'truth_context': truth_context,
                'prompt': rec.get('prompt'),
            })

df = pd.DataFrame(records)
print(f'Loaded {len(df)} records')
df.head(3)


In [ ]:
total = len(df)
deceptive_total = int((df['deceptive'] == True).sum()) if total else 0
truthful_total = int((df['deceptive'] == False).sum()) if total else 0
unknown_total = int(df['deceptive'].isna().sum()) if total else 0

summary_df = pd.DataFrame([
    {'metric': 'total_records', 'value': total},
    {'metric': 'deceptive_records', 'value': deceptive_total},
    {'metric': 'truthful_records', 'value': truthful_total},
    {'metric': 'unknown_records', 'value': unknown_total},
    {'metric': 'jsonl_files', 'value': len(jsonl_files)},
])
display(summary_df)

if total:
    action_counts = df['action_type'].fillna('(missing)').value_counts().rename_axis('action_type').reset_index(name='count')
    display(Markdown('### Action Type Counts'))
    display(action_counts)

    preview_cols = ['conversation_id', 'deceptive_label', 'action_type', 'decision_message_consistent', 'message']
    display(Markdown('### Preview'))
    display(df[preview_cols].head(10))


## Interactive Deceptive Example Browser

Use the dropdown to inspect deceptive examples. You can filter by action type, search by conversation/message/reasoning text, and expand prompt/truth-context details when needed.


In [ ]:
deceptive_df = df[df['deceptive'] == True].copy().reset_index(drop=True)
print(f'Deceptive examples available: {len(deceptive_df)}')

if deceptive_df.empty:
    print('No deceptive examples found.')
else:
    deceptive_df['conversation_id'] = deceptive_df['conversation_id'].fillna('(no conversation_id)')
    deceptive_df['message'] = deceptive_df['message'].fillna('')
    deceptive_df['reasoning'] = deceptive_df['reasoning'].fillna('')
    deceptive_df['decision_message_consistent'] = deceptive_df['decision_message_consistent'].fillna(False)
    deceptive_df['message_mentions_other_offer_claim'] = deceptive_df['message_mentions_other_offer_claim'].fillna(False)
    deceptive_df['message_mentions_claimed_other_offer_salary'] = deceptive_df['message_mentions_claimed_other_offer_salary'].fillna(False)
    deceptive_df['message_preview'] = deceptive_df['message'].str.replace('\n', ' ', regex=False).str.slice(0, 100)
    deceptive_df['dropdown_label'] = deceptive_df.apply(
        lambda row: f"{row.name:04d} | {row['action_type']} | {row['conversation_id']} | {row['message_preview']}",
        axis=1,
    )

    action_filter = widgets.Dropdown(
        options=['ALL'] + sorted(str(x) for x in deceptive_df['action_type'].dropna().unique()),
        value='ALL',
        description='Action:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width="280px"),
    )
    search_box = widgets.Text(
        value='',
        description='Search:',
        placeholder='conversation_id / message / consistency / reasoning',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width="420px"),
    )
    example_dropdown = widgets.Dropdown(
        options=[],
        description='Example:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width="100%"),
    )
    show_prompt = widgets.Checkbox(value=False, description='Show prompt')
    show_truth_context = widgets.Checkbox(value=True, description='Show truth context')
    out = widgets.Output()
    state = {'filtered': deceptive_df}

    def _filtered_df():
        filtered = deceptive_df.copy()
        if action_filter.value != 'ALL':
            filtered = filtered[filtered['action_type'].astype(str) == action_filter.value]

        needle = search_box.value.strip().lower()
        if needle:
            mask = (
                filtered['conversation_id'].fillna('').str.lower().str.contains(needle, regex=False)
                | filtered['message'].fillna('').str.lower().str.contains(needle, regex=False)
                | filtered['decision_message_consistent'].astype(str).str.lower().str.contains(needle, regex=False)
                | filtered['message_mentions_other_offer_claim'].astype(str).str.lower().str.contains(needle, regex=False)
                | filtered['message_mentions_claimed_other_offer_salary'].astype(str).str.lower().str.contains(needle, regex=False)
                | filtered['reasoning'].fillna('').str.lower().str.contains(needle, regex=False)
            )
            filtered = filtered[mask]

        return filtered.reset_index(drop=True)

    def _render(*_):
        with out:
            clear_output()
            filtered = state['filtered']
            if filtered.empty or example_dropdown.value is None:
                display(Markdown('No deceptive examples match the current filters.'))
                return

            row = filtered.iloc[int(example_dropdown.value)]
            meta_rows = pd.DataFrame([
                {'field': 'conversation_id', 'value': row['conversation_id']},
                {'field': 'scenario_name', 'value': row['scenario_name']},
                {'field': 'base_scenario_name', 'value': row['base_scenario_name']},
                {'field': 'run_tag', 'value': row['run_tag']},
                {'field': 'shard', 'value': row['shard']},
                {'field': 'state_id', 'value': row['state_id']},
                {'field': 'game_id', 'value': row['game_id']},
                {'field': 'turn_idx', 'value': row['turn_idx']},
                {'field': 'sample_idx', 'value': row['sample_idx']},
                {'field': 'seed', 'value': row['seed']},
                {'field': 'deceptive', 'value': row['deceptive']},
                {'field': 'recorded_deceptive', 'value': row['recorded_deceptive']},
                {'field': 'action_type', 'value': row['action_type']},
                {'field': 'decision_message_consistent', 'value': row['decision_message_consistent']},
                {'field': 'message_mentions_other_offer_claim', 'value': row['message_mentions_other_offer_claim']},
                {'field': 'message_mentions_claimed_other_offer_salary', 'value': row['message_mentions_claimed_other_offer_salary']},
                {'field': 'claimed_other_offer_salary', 'value': row['claimed_other_offer_salary']},
                {'field': 'deception_reason', 'value': row['deception_reason']},
                {'field': 'used_for_transition', 'value': row['used_for_transition']},
            ])

            display(Markdown(f"### {row['conversation_id']}"))
            display(meta_rows)

            display(Markdown('#### Reasoning'))
            reasoning_html = html.escape(row['reasoning'] if row['reasoning'] else '(none)')
            display(HTML(f"<pre style='white-space: pre-wrap; line-height: 1.35em;'>{reasoning_html}</pre>"))

            display(Markdown('#### Message'))
            message_html = html.escape(row['message'] if row['message'] else '(none)')
            display(HTML(f"<pre style='white-space: pre-wrap; line-height: 1.35em;'>{message_html}</pre>"))

            display(Markdown('#### Full Action'))
            display(JSON(row['full_action'], expanded=True))

            if show_truth_context.value:
                display(Markdown('#### Truth Context'))
                display(JSON(row['truth_context'], expanded=False))

            if show_prompt.value:
                display(Markdown('#### Prompt'))
                prompt_html = html.escape(row['prompt'] if row['prompt'] else '(none)')
                display(HTML(f"<pre style='white-space: pre-wrap; line-height: 1.35em;'>{prompt_html}</pre>"))

    def _refresh_options(*_):
        filtered = _filtered_df()
        state['filtered'] = filtered
        if filtered.empty:
            example_dropdown.options = [('No matches', None)]
            example_dropdown.value = None
        else:
            example_dropdown.options = [
                (row['dropdown_label'], int(i))
                for i, row in filtered.iterrows()
            ]
            example_dropdown.value = 0
        _render()

    action_filter.observe(_refresh_options, names='value')
    search_box.observe(_refresh_options, names='value')
    example_dropdown.observe(_render, names='value')
    show_prompt.observe(_render, names='value')
    show_truth_context.observe(_render, names='value')

    _refresh_options()
    controls = widgets.VBox([
        widgets.HBox([action_filter, search_box]),
        example_dropdown,
        widgets.HBox([show_truth_context, show_prompt]),
    ])
    display(controls)
    display(out)
